# File 1
### This notebook cleans the harmonised demographics and links them to permission status



In [ ]:
import pandas as pd
import numpy as np



In [ ]:
# permissions
df_perm = pd.read_stata("S:\LLC_0002\lamj\Datasets\Participant Base\Permission_Status_Base.dta")


In [ ]:
# remove if no permission to use as part of UK LLC
df_perm = df_perm[df_perm['ukllc_status'] == 1]

In [ ]:
df_perm.rename(columns = {'gender': 'gender_perm'}, inplace = True)

In [ ]:
# linking with demographic variables (dated)

bestMeasure_sex = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_Sex.dta')
bestMeasure_ethnic7 = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_ethnic7.dta')
bestMeasure_age = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_age.dta')
bestMeasure_gender = pd.read_stata(r'S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\NHS_&_Self-Report_BestMeasure_Gender.dta')

merged_sex = pd.merge(df_perm,bestMeasure_sex[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id", how = 'left')
merged_sex.rename(columns = {'value': 'sex'}, inplace = True)
merged_sex['sex'].replace({8:"Male",9:"Female",10:"Non-binary",11:"Self-define", 12:"Prefer not to answer", 99:pd.NA}, inplace = True)
merged_sex.drop(columns=['label'], inplace = True)

merged_sex_gender = pd.merge(merged_sex,bestMeasure_gender[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id", how = 'left')
merged_sex_gender.rename(columns = {'value': 'gender'}, inplace = True)
merged_sex_gender['gender'].replace({8:"Male",9:"Female",10:"Non-binary",11:"Self-define", 12:"Prefer not to answer",99:pd.NA}, inplace = True)
merged_sex_gender.drop(columns=['label'], inplace = True)

merged_sex_ethnic = pd.merge(merged_sex_gender,bestMeasure_ethnic7[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id", how = 'left')
merged_sex_ethnic.rename(columns = {'value': 'ethnicity'}, inplace = True)
merged_sex_ethnic['ethnicity'].replace({0:"White",1:"Black",2:"South-east Asian",3:"Other Asian", 4:"Mixed",5:"Other",99:pd.NA}, inplace = True)
merged_sex_ethnic.drop(columns=['label'], inplace = True)

merged_sex_ethnic_age = pd.merge(merged_sex_ethnic,bestMeasure_age[['LLC_0002_stud_id', "value", 'label']], on = "LLC_0002_stud_id",how = 'left')
merged_sex_ethnic_age.rename(columns = {'value': 'age'}, inplace = True)
merged_sex_ethnic_age['age'].replace({13:"<=18 years",14:"19-30 years",15:"31-59 years",16:"60-74 years", 17:"75+ years"}, inplace = True)
merged_sex_ethnic_age.drop(columns=['label'], inplace = True)

merged_df = merged_sex_ethnic_age
merged_df = merged_df.rename(columns = {"LLC_0002_stud_id": "llc_0002_stud_id"})



In [ ]:
"""merged_df['sex_or_gender']  = merged_df['sex'].fillna(merged_df['gender'])
merged_df[['sex', 'gender', 'sex_or_gender']].head(5)"""


In [ ]:
# identify mismatched... could be of use 
merged_df['gender_mismatch_flag'] = (merged_df['sex'] != merged_df['gender']) & (merged_df['sex'].notna() & merged_df['gender'].notna())
flagged = merged_df[merged_df['gender_mismatch_flag']== True]
tab = pd.crosstab(flagged['sex'], flagged['gender'], margins = True, margins_name = "Total")
tab


In [ ]:
merged_df.columns

In [ ]:
# data reduction - not including sex + gender
columns = ['llc_0002_stud_id', "cohort", 'ukllc_status', 'nhs_e_linkage_permission', 'nhs_ni_linkage_permission',
       'nhs_s_linkage_permission', 'nhs_w_linkage_permission', 
           'NHSD_linked', 'national_opt_out', 
           'death_notification_status', 'derived_for_dodym',"dob_year_month",
          'sex', 'gender','ethnicity','age']#'sex_or_gender',]
df_demo = merged_df[columns]



In [ ]:
# clean gender - based on cis/trans

def classify_gender_identity(row):
    sex = row['sex']
    gender = row['gender']
    
    if pd.isna(sex) or pd.isna(gender):
        return pd.NA
    elif sex == gender:
        return 'cis-gender'
    elif (sex == 'Male' and gender == 'Female') or (sex == 'Female' and gender == 'Male'):
        return 'transgender'

df_demo['gender_identity'] = df_demo.apply(classify_gender_identity, axis = 1)

In [ ]:
df_demo.loc[df_demo['gender_identity'].isin(['Male','Female']), 'gender_identity'] = pd.NA

In [ ]:
df_demo['gender_identity'].value_counts()

In [ ]:
# combine prefer not to answer to missing
df_demo.loc[df_demo['gender'].isin(['Prefer not to answer']), 'gender'] = pd.NA
df_demo.loc[df_demo['gender_identity'].isin(['Prefer not to answer']), 'gender_identity'] = pd.NA
# fillnas
df_demo[['sex','gender', 'gender_identity','ethnicity','age']].value_counts(dropna=False)
df_demo['age'] = df_demo['age'].fillna("Missing")
df_demo['ethnicity'] = df_demo['ethnicity'].fillna("Missing")
df_demo['sex'] = df_demo['sex'].fillna("Missing")
df_demo['gender'] = df_demo['gender'].fillna("Missing")
df_demo['gender_identity'] = df_demo['gender_identity'].fillna("Missing")
df_demo['NHSD_linked'] = df_demo['NHSD_linked'].fillna("Not Linked")




In [ ]:
# clean consent type
conditions = [
    (df_demo['nhs_e_linkage_permission'] == "0"),
    (df_demo['nhs_e_linkage_permission'] == "1") & (df_demo['national_opt_out'] == 1),
    (df_demo['nhs_e_linkage_permission'] == "1") & (df_demo['national_opt_out'] == 0)    
]
    
values = ["Dissent", "S251", "Consent"]
df_demo['consent_type'] = np.select(conditions, values, default = 0)


In [ ]:
# keep fewer columns
columns = ['llc_0002_stud_id', "cohort", 'ukllc_status',
           'NHSD_linked', "consent_type",
           'death_notification_status', 'derived_for_dodym',"dob_year_month",
          'sex', 'gender','gender_identity','ethnicity','age']#'sex_or_gender',]
df_demo = df_demo[columns]

In [ ]:
df_demo

In [ ]:
df_demo.to_csv(r"S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\demographics_LLC.csv")

In [ ]:
df_demo['cohort'].value_counts(dropna=False)

In [ ]:
# create English subset, to exclude GENSCOT, NICOLA, SABRE
df_demo_english = df_demo[~(df_demo['cohort'] == "GENSCOT") & ~(df_demo['cohort'] == "NICOLA") & ~(df_demo['cohort'] == "SABRE")]


In [ ]:
df_noperm = df_demo_english[df_demo_english['consent_type'] == "Dissent"]

df_noperm['gender'].value_counts(dropna = False)


In [ ]:
df_noperm

In [ ]:
df_demo['gender'].value_counts()

In [ ]:
# Load TableOne Package
# Create demographics for NO permission vs With Permission to link.
from tableone import TableOne

# no perm
columns = ['sex', 'gender','gender_identity','ethnicity', 'age']
categorical = ['sex','gender', 'gender_identity','ethnicity', 'age']
order = {"age":["<=18 years","19-30 years","31-59 years","60-74 years","75+ years", "Missing"],
         "sex":['Female', 'Male', 'Missing'],
         'gender': ['Female','Male','Non-binary','Self-define'],
         'gender_identity': ['cis-gender', 'transgender', 'Non-binary','Self-define', 'Missing'],
        "ethnicity":["Missing", "White", "South-east Asian", "Black", "Other", "Mixed", "Other Asian"]}
noperm_table = TableOne(df_noperm, columns, categorical,order = order)
noperm_table.tableone


In [ ]:
# row-wise percentage
categorical = ['sex','ethnicity', 'age']

for var in categorical:
    dist = df_noperm[var].value_counts(dropna = False, normalize = True) * 100
    print(f"\nDistribution for {var}:\n")
    print(dist.round(2).to_frame(name ='percentage'))
    


In [ ]:
# S251

df_s251 = df_demo_english[(df_demo_english['consent_type'] == "S251")]
df_s251['linked']  = df_s251['NHSD_linked'].apply(lambda x: "Not Linked" if x == "Not Linked" else "Linked")

columns = ['sex', 'gender','gender_identity','ethnicity', 'age']
categorical = ['sex','gender', 'gender_identity','ethnicity', 'age']
order = {"age":["<=18 years","19-30 years","31-59 years","60-74 years","75+ years", "Missing"],
         "sex":['Female', 'Male', 'Missing'],
         'gender': ['Female','Male','Non-binary','Self-define'],
         'gender_identity': ['cis-gender', 'transgender', 'Non-binary','Self-define', 'Missing'],
        "ethnicity":["Missing", "White", "South-east Asian", "Black", "Other", "Mixed", "Other Asian"]}

s251_table = TableOne(df_s251, columns, categorical, order = order, groupby = "linked")
s251_table.tableone

In [ ]:
# row-wise percentage
categorical = ['sex','ethnicity', 'age']

for var in categorical:
    rowwise = pd.crosstab(df_s251[var], df_s251['linked'], normalize = 'index') * 100
    rowwise.columns = [f'treatment={col}' for col in rowwise.columns]
    
    row_counts = pd.crosstab(df_s251[var], df_s251['linked']).sum(axis = 1).rename('Total')
    
    rowwise = rowwise.join(row_counts)
    rowwise.reset_index(inplace = True)
    print(f"\nDistribution for {var}:\n")
    print(rowwise)

In [ ]:
# proportion standardised differences

def smd_categorical(df, treatment_col, categorical_vars):
    results = []
    
    for var in categorical_vars:
        group1 = df[df[treatment_col] == 1][var].dropna()
        group0 = df[df[treatment_col] == 0][var].dropna()
        
        # get unique levels across groups
        levels = sorted(set(group1.unique()).union(set(group0.unique())))
        
        for level in levels:
            if (len(group1) == 0  or len(group0) == 0):
                sd = np.nan
                p1 = np.nan
                p0 = np.nan
            else:
                p1 = np.mean(group1 == level)
                p0 = np.mean(group0 == level)
                pooled_var = (p1 * (1 - p1) + p0 * (1-p0)) / 2
                sd = (p1-p0) /np.sqrt(pooled_var) if pooled_var > 0 else 0
            
            results.append({
                'Variable':var,
                'Level': level,
                'Linked (p1)': round(p1, 6) if not pd.isna(p1) else np.nan,
                'Not Linked (p0)': round(p0, 6) if not pd.isna(p0) else np.nan,
                'Standardized Difference': round(sd, 6) if not pd.isna(p1) else np.nan
            })
    return pd.DataFrame(results)



In [ ]:
df_s251['linked'].value_counts()

In [ ]:
map = {
    'Linked': 1,
    'Not Linked': 0,
}

df_s251['linked_num'] = df_s251['linked'].map(map)

In [ ]:
categorical_vars = ['sex', 'ethnicity', 'age']
df_s251_results = smd_categorical(df_s251, treatment_col = 'linked_num', categorical_vars = categorical_vars)

In [ ]:
df_s251_results

In [ ]:
# n x n fisher exact test - monte carlo 
import numpy as np
import itertools
import scipy.stats as stats
import random

def fisher_exact_monte_carlo(table, num_samples = 10000):
    table = np.asarray(table)
    
    if table.ndim != 2:
        raise ValueError("input table must be 2D array or list of lists")
    
    observed_stat = stats.chi2_contingency(table, correction = False)[0]
    total = np.sum(table)
    row_sums = np.sum(table, axis = 1)
    col_sums = np.sum(table, axis = 0)
    
    # generate count
    count = 0 
    for _ in range(num_samples):
        randomised_table = np.zeros_like(table)
        for i, row_sum in enumerate(row_sums):
            randomised_tabel[i, :] = np.random.multinomial(row_sum, col_sums / total)
            
        rand_stat = stats.chi2_contingency(randomised_table, correction = False)[0]
        
        if rnad_stat >= observed_stat:
            count += 1
    
    p_value = count/ num_samples
    return p_value

custom_tests = {'sex_or_gender': fisher_exact_monte_carlo, 'ethnicity': fisher_exact_monte_carlo, 'age': fisher_exact_monte_carlo}

In [ ]:
# inspect demographics by S251, Consent, linked, unlinked.

df_consent = df_demo_english[(df_demo_english['consent_type'] == "Consent")]
df_consent['linked']  = df_consent['NHSD_linked'].apply(lambda x: "Not Linked" if x == "Not Linked" else "Linked")

columns = ['sex', 'gender','gender_identity','ethnicity', 'age']
categorical = ['sex','gender', 'gender_identity','ethnicity', 'age']
order = {"age":["<=18 years","19-30 years","31-59 years","60-74 years","75+ years", "Missing"],
         "sex":['Female', 'Male', 'Missing'],
         'gender': ['Female','Male','Non-binary','Self-define'],
         'gender_identity': ['cis-gender', 'transgender', 'Non-binary','Self-define', 'Missing'],
        "ethnicity":["Missing", "White", "South-east Asian", "Black", "Other", "Mixed", "Other Asian"]}

df_consent_table = TableOne(df_consent, columns, categorical, order = order, groupby = "linked", pval = True, htest_name = True)
df_consent_table.tableone




In [ ]:
# row-wise percentage
categorical = ['sex','ethnicity', 'age']

for var in categorical:
    rowwise = pd.crosstab(df_consent[var], df_consent['linked'], normalize = 'index') * 100
    rowwise.columns = [f'treatment={col}' for col in rowwise.columns]
    
    row_counts = pd.crosstab(df_consent[var], df_consent['linked']).sum(axis = 1).rename('Total')
    
    rowwise = rowwise.join(row_counts)
    rowwise.reset_index(inplace = True)
    print(f"\nDistribution for {var}:\n")
    print(rowwise)

In [ ]:
df_consent = df_demo_english[(df_demo_english['consent_type'] == "Consent")]
df_consent['linked']  = df_consent['NHSD_linked'].apply(lambda x: "Not Linked" if x == "Not Linked" else "Linked")

In [ ]:
df_consent

In [ ]:
map = {
    'Linked': 1,
    'Not Linked': 0,
}

df_consent['linked_num'] = df_consent['linked'].map(map)

In [ ]:
categorical_vars = ['sex', 'ethnicity', 'age']
df_consent_results = smd_categorical(df_consent, treatment_col = 'linked_num', categorical_vars = categorical_vars)

In [ ]:
df_consent_results

In [ ]:
# finalising: linkage eval cohort - only include with permission

df_demo_english
df_demo_english.to_csv("English_LLC_cohort.csv")

In [ ]:
df_demo_english

In [ ]:
df_demo_english['consent_type'] != "Dissent"

In [ ]:
df_demo_english['NHSD_linked'] != "Not Linked"

In [ ]:
linkage_cohort = df_demo_english[(df_demo_english['NHSD_linked'] != "Not Linked") & (df_demo_english['consent_type'] != "Dissent")]

In [ ]:
linkage_cohort

In [ ]:
linkage_cohort.to_csv("English_LLC_cohort.csv")